# CMPS 355 — Phase 2 Baselines v3 (C1): Hardened Fail-Closed, Checkpointed & Stage 22 Parity

**GATED V3 STAGING NOTEBOOK — DO NOT ENABLE `RUN_SMOKE_TEST` OR `ALLOW_FULL_RUN` UNTIL CHATGPT INDEPENDENT V3 SIGN-OFF PASSES.**

- Expected `phase2_baselines_v3.py` SHA-256: `14387756ef1c85f0a2bf8c1cf0a108bc2575cec645365ae180e48b9c7ce6a1e1`
- Immutable canonical ZIP: `/content/drive/MyDrive/CMPS355_Research/data/validated/ibd_dataset_sample_aligned.zip` (`SHA-256: 8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74`)
- Exact Stage 22 parity: `LogisticRegression(penalty="l2", C=1.0, class_weight="balanced", solver="liblinear", max_iter=5000, random_state=123)` (sklearn default `tol=1e-4`), `CPM > 1.0` in `>= 10%` of training samples.
- Note: Live R `limma-voom` has **not** yet been run in Colab. Cell 2 (`RUN_SMOKE_TEST=False`) runs a tiny D3 smoke test (`SMOKE_ONLY_NOT_PRODUCTION`) only after ChatGPT authorizes v3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, subprocess, sys
from pathlib import Path

STAGE_DIR = Path('/content/drive/MyDrive/CMPS355_Research/peer_review_revision_2026/03_followup_experiments/phase2_baselines_20261122_v3')
SCRIPT_PATH = STAGE_DIR / 'phase2_baselines_v3.py'
DATA_ZIP = Path('/content/drive/MyDrive/CMPS355_Research/data/validated/ibd_dataset_sample_aligned.zip')
OUT_DIR = STAGE_DIR / 'outputs'
SMOKE_DIR = STAGE_DIR / 'smoke_test_d3'
EXPECTED_SCRIPT_SHA = '14387756ef1c85f0a2bf8c1cf0a108bc2575cec645365ae180e48b9c7ce6a1e1'
EXPECTED_DATA_SHA = '8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

assert SCRIPT_PATH.is_file(), f'Missing script: {SCRIPT_PATH}'
assert sha256_file(SCRIPT_PATH) == EXPECTED_SCRIPT_SHA, 'Script SHA-256 mismatch'
assert DATA_ZIP.is_file(), f'Canonical ZIP missing: {DATA_ZIP}'
assert sha256_file(DATA_ZIP) == EXPECTED_DATA_SHA, 'Dataset ZIP SHA-256 mismatch'
print('Verified C1 v3 script and canonical ZIP SHA-256 hashes.')

r_expr = 'cat(R.version.string, "\n"); suppressPackageStartupMessages({library(limma); library(edgeR)}); cat("limma=", as.character(packageVersion("limma")), " edgeR=", as.character(packageVersion("edgeR")), "\n", sep="")'
r_check = subprocess.run(['Rscript', '-e', r_expr], capture_output=True, text=True)
print(r_check.stdout)
if r_check.returncode != 0:
    r_inst = 'if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager", repos="https://cloud.r-project.org"); BiocManager::install(c("limma", "edgeR"), update=FALSE, ask=FALSE)'
    subprocess.run(['Rscript', '-e', r_inst], check=True)
    subprocess.run(['Rscript', '-e', r_expr], check=True)


In [ ]:
# Cell 2: Tiny live R limma-voom smoke test (D3 fold only, B=5, n_boot=200) -> SMOKE_ONLY_NOT_PRODUCTION
RUN_SMOKE_TEST = True
if not RUN_SMOKE_TEST:
    raise RuntimeError('BLOCKED: RUN_SMOKE_TEST=False. Enable only after ChatGPT approves v3.')
assert sha256_file(SCRIPT_PATH) == EXPECTED_SCRIPT_SHA and sha256_file(DATA_ZIP) == EXPECTED_DATA_SHA
import time
t0 = time.time()
subprocess.run([sys.executable, str(SCRIPT_PATH), '--data', str(DATA_ZIP), '--out', str(SMOKE_DIR), '--smoke-test', '--n-random', '5', '--n-boot', '200'], check=True)
print(f'Live D3 smoke test finished in {time.time() - t0:.2f}s (SMOKE_ONLY_NOT_PRODUCTION)')


In [ ]:
# Cell 3: FULL PHASE 2 BASELINES V3 (C1) — STRICTLY GATED
ALLOW_FULL_RUN = True
if not ALLOW_FULL_RUN:
    raise RuntimeError('BLOCKED: ALLOW_FULL_RUN=False. Requires explicit ChatGPT authorization.')
assert sha256_file(SCRIPT_PATH) == EXPECTED_SCRIPT_SHA and sha256_file(DATA_ZIP) == EXPECTED_DATA_SHA
import time
t0 = time.time()
subprocess.run([sys.executable, str(SCRIPT_PATH), '--data', str(DATA_ZIP), '--out', str(OUT_DIR), '--n-random', '100', '--n-boot', '2000'], check=True)
print(f'Full C1 v3 finished in {time.time() - t0:.2f}s')


In [ ]:
# Cell 4: Fail-closed --finalize-only verification
RUN_FINALIZE_ONLY = False
if not RUN_FINALIZE_ONLY:
    raise RuntimeError('BLOCKED: RUN_FINALIZE_ONLY=False.')
assert sha256_file(SCRIPT_PATH) == EXPECTED_SCRIPT_SHA
subprocess.run([sys.executable, str(SCRIPT_PATH), '--out', str(OUT_DIR), '--finalize-only'], check=True)
